In [1]:
# ============================================================
# WEEK 6 - CUSTOMER CHURN
# Imbalance + GridSearchCV + Recall + Explainability
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
df = pd.read_csv("Customer-Churn.csv")

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.shape

(7043, 21)

In [5]:
# ------------------------------------------------------------
# 2. CLEAN DATA
# ------------------------------------------------------------

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Convert Churn into 0 and 1
# No = 0
# Yes = 1
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [6]:
# Remove customerID because it is only an identifier
df = df.drop("customerID", axis=1)


In [7]:
# ------------------------------------------------------------
# 3. CHECK CLASS IMBALANCE
# ------------------------------------------------------------

print("\nChurn distribution:")
print(df["Churn"].value_counts())

print("\nChurn percentage:")
print(df["Churn"].value_counts(normalize=True) * 100)


Churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn percentage:
Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64


In [8]:
# ------------------------------------------------------------
# 4. SEPARATE X AND y
# ------------------------------------------------------------

X = df.drop("Churn", axis=1)
y = df["Churn"]

In [9]:
# ------------------------------------------------------------
# 5. IDENTIFY NUMERIC AND CATEGORICAL FEATURES
# ------------------------------------------------------------

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns

print("\nNumeric features:")
print(list(numeric_features))

print("\nCategorical features:")
print(list(categorical_features))


Numeric features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\Arati\AppData\Local\Temp\ipykernel_10644\227127208.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [10]:
# ------------------------------------------------------------
# 6. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 5634
Testing samples: 1409


In [11]:
# ------------------------------------------------------------
# 7. PREPROCESSING
# ------------------------------------------------------------

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [12]:
# ------------------------------------------------------------
# 8. BASELINE MODEL
# ------------------------------------------------------------

baseline_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_recall = recall_score(
    y_test,
    baseline_pred
)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)
print("\n====================================")
print("BASELINE MODEL")
print("====================================")

print("Baseline Accuracy:",
      round(baseline_accuracy, 4))

print("Baseline Recall:",
      round(baseline_recall, 4))

print("\nBaseline Classification Report:")
print(classification_report(
    y_test,
    baseline_pred,
    target_names=["No Churn", "Churn"]
))



BASELINE MODEL
Baseline Accuracy: 0.8055
Baseline Recall: 0.5588

Baseline Classification Report:
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1035
       Churn       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [19]:
# ------------------------------------------------------------
# 9. GRID SEARCH
# ------------------------------------------------------------

# We give more importance to the minority class
# class_weight='balanced' automatically gives higher weight
# to the class having fewer samples.

tuned_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])


# Hyperparameters we want to test
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],

    "model__class_weight": [
        None,
        "balanced",
        {0: 1, 1: 1.5},
        {0: 1, 1: 2},
        {0: 1, 1: 3}
    ],

    "model__solver": [
        "liblinear",
        "lbfgs"
    ]
}


# StratifiedKFold keeps the churn/no-churn ratio similar
# in every fold.

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# IMPORTANT:
# We optimize RECALL, not accuracy.
grid_search = GridSearchCV(
    estimator=tuned_pipeline,
    param_grid=param_grid,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV Recall:")
print(grid_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters:
{'model__C': 100, 'model__class_weight': {0: 1, 1: 3}, 'model__solver': 'lbfgs'}
Best CV Recall:
0.8167224080267559


In [20]:
# ------------------------------------------------------------
# 10. BEST PARAMETERS
# ------------------------------------------------------------

print("\n====================================")
print("BEST PARAMETERS")
print("====================================")

print(grid_search.best_params_)

print("\nBest Cross-Validation Recall:")
print(round(grid_search.best_score_, 4))


BEST PARAMETERS
{'model__C': 100, 'model__class_weight': {0: 1, 1: 3}, 'model__solver': 'lbfgs'}

Best Cross-Validation Recall:
0.8167


In [21]:
# ------------------------------------------------------------
# 11. TEST THE TUNED MODEL
# ------------------------------------------------------------

best_model = grid_search.best_estimator_

tuned_pred = best_model.predict(X_test)

tuned_accuracy = accuracy_score(
    y_test,
    tuned_pred
)

tuned_precision = precision_score(
    y_test,
    tuned_pred
)

tuned_recall = recall_score(
    y_test,
    tuned_pred
)

tuned_f1 = f1_score(
    y_test,
    tuned_pred
)


print("\n====================================")
print("TUNED MODEL RESULTS")
print("====================================")

print("Accuracy :", round(tuned_accuracy, 4))
print("Precision:", round(tuned_precision, 4))
print("Recall   :", round(tuned_recall, 4))
print("F1 Score :", round(tuned_f1, 4))


TUNED MODEL RESULTS
Accuracy : 0.7282
Precision: 0.4926
Recall   : 0.8021
F1 Score : 0.6104


In [22]:
# ------------------------------------------------------------
# 12. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    tuned_pred
)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    tuned_pred,
    target_names=["No Churn", "Churn"]
))



Confusion Matrix:
[[726 309]
 [ 74 300]]

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.91      0.70      0.79      1035
       Churn       0.49      0.80      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.80      0.73      0.74      1409



In [23]:
# ------------------------------------------------------------
# 13. COMPARE BASELINE VS TUNED
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Model": [
        "Baseline Logistic Regression",
        "Tuned Logistic Regression"
    ],

    "Accuracy": [
        baseline_accuracy,
        tuned_accuracy
    ],

    "Precision": [
        precision_score(y_test, baseline_pred),
        tuned_precision
    ],

    "Recall": [
        baseline_recall,
        tuned_recall
    ],

    "F1 Score": [
        f1_score(y_test, baseline_pred),
        tuned_f1
    ]
})

print("\n====================================")
print("BASELINE VS TUNED")
print("====================================")

print(comparison.round(4))



BASELINE VS TUNED
                          Model  Accuracy  Precision  Recall  F1 Score
0  Baseline Logistic Regression    0.8055     0.6572  0.5588    0.6040
1     Tuned Logistic Regression    0.7282     0.4926  0.8021    0.6104


In [24]:
# ------------------------------------------------------------
# 14. CHECK WHETHER RECALL IMPROVED
# ------------------------------------------------------------

recall_change = tuned_recall - baseline_recall

print("\n====================================")
print("RECALL IMPROVEMENT")
print("====================================")

print(
    "Baseline Recall:",
    round(baseline_recall, 4)
)

print(
    "Tuned Recall:",
    round(tuned_recall, 4)
)

print(
    "Recall Improvement:",
    round(recall_change, 4)
)


if tuned_recall > baseline_recall:
    print("\nSUCCESS: Recall improved!")
else:
    print("\nRecall did not improve. Further tuning may be needed.")


RECALL IMPROVEMENT
Baseline Recall: 0.5588
Tuned Recall: 0.8021
Recall Improvement: 0.2433

SUCCESS: Recall improved!


In [25]:
# ------------------------------------------------------------
# 15. EXPLAINABILITY
# ------------------------------------------------------------

# Get transformed feature names
feature_names = best_model.named_steps[
    "preprocessing"
].get_feature_names_out()


# Get Logistic Regression coefficients
coefficients = best_model.named_steps[
    "model"
].coef_[0]


feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})


# Positive coefficient = pushes prediction toward Churn
# Negative coefficient = pushes prediction toward No Churn

feature_importance["Direction"] = np.where(
    feature_importance["Coefficient"] > 0,
    "Higher Churn Risk",
    "Lower Churn Risk"
)


In [26]:
# ------------------------------------------------------------
# 16. TOP FEATURES PUSHING TOWARD CHURN
# ------------------------------------------------------------

top_churn_features = feature_importance.sort_values(
    by="Coefficient",
    ascending=False
).head(15)

print("\n====================================")
print("TOP FEATURES PUSHING TOWARD CHURN")
print("====================================")

print(
    top_churn_features[
        ["Feature", "Coefficient", "Direction"]
    ].to_string(index=False)
)



TOP FEATURES PUSHING TOWARD CHURN
                            Feature  Coefficient         Direction
   cat__InternetService_Fiber optic     1.579705 Higher Churn Risk
           cat__StreamingMovies_Yes     0.699145 Higher Churn Risk
               cat__StreamingTV_Yes     0.685000 Higher Churn Risk
       cat__Contract_Month-to-month     0.625955 Higher Churn Risk
                  num__TotalCharges     0.519690 Higher Churn Risk
             cat__MultipleLines_Yes     0.437206 Higher Churn Risk
          cat__DeviceProtection_Yes     0.345101 Higher Churn Risk
              cat__OnlineBackup_Yes     0.288865 Higher Churn Risk
              cat__PhoneService_Yes     0.231409 Higher Churn Risk
cat__PaymentMethod_Electronic check     0.206087 Higher Churn Risk
               cat__TechSupport_Yes     0.194559 Higher Churn Risk
             cat__OnlineSecurity_No     0.172096 Higher Churn Risk
            cat__OnlineSecurity_Yes     0.157059 Higher Churn Risk
                cat__TechSu

In [27]:
# ------------------------------------------------------------
# 16. TOP FEATURES PUSHING TOWARD CHURN
# ------------------------------------------------------------

top_churn_features = feature_importance.sort_values(
    by="Coefficient",
    ascending=False
).head(15)

print("\n====================================")
print("TOP FEATURES PUSHING TOWARD CHURN")
print("====================================")

print(
    top_churn_features[
        ["Feature", "Coefficient", "Direction"]
    ].to_string(index=False)
)


TOP FEATURES PUSHING TOWARD CHURN
                            Feature  Coefficient         Direction
   cat__InternetService_Fiber optic     1.579705 Higher Churn Risk
           cat__StreamingMovies_Yes     0.699145 Higher Churn Risk
               cat__StreamingTV_Yes     0.685000 Higher Churn Risk
       cat__Contract_Month-to-month     0.625955 Higher Churn Risk
                  num__TotalCharges     0.519690 Higher Churn Risk
             cat__MultipleLines_Yes     0.437206 Higher Churn Risk
          cat__DeviceProtection_Yes     0.345101 Higher Churn Risk
              cat__OnlineBackup_Yes     0.288865 Higher Churn Risk
              cat__PhoneService_Yes     0.231409 Higher Churn Risk
cat__PaymentMethod_Electronic check     0.206087 Higher Churn Risk
               cat__TechSupport_Yes     0.194559 Higher Churn Risk
             cat__OnlineSecurity_No     0.172096 Higher Churn Risk
            cat__OnlineSecurity_Yes     0.157059 Higher Churn Risk
                cat__TechSu

In [28]:
# ------------------------------------------------------------
# 17. TOP FEATURES REDUCING CHURN RISK
# ------------------------------------------------------------

top_safe_features = feature_importance.sort_values(
    by="Coefficient",
    ascending=True
).head(15)

print("\n====================================")
print("TOP FEATURES REDUCING CHURN RISK")
print("====================================")

print(
    top_safe_features[
        ["Feature", "Coefficient", "Direction"]
    ].to_string(index=False)
)


TOP FEATURES REDUCING CHURN RISK
                                  Feature  Coefficient        Direction
                      num__MonthlyCharges    -2.497048 Lower Churn Risk
                 cat__InternetService_DSL    -1.250550 Lower Churn Risk
                              num__tenure    -1.177708 Lower Churn Risk
                   cat__Contract_Two year    -0.837180 Lower Churn Risk
cat__DeviceProtection_No internet service    -0.648988 Lower Churn Risk
    cat__OnlineBackup_No internet service    -0.648988 Lower Churn Risk
  cat__OnlineSecurity_No internet service    -0.648988 Lower Churn Risk
     cat__TechSupport_No internet service    -0.648988 Lower Churn Risk
     cat__StreamingTV_No internet service    -0.648988 Lower Churn Risk
                  cat__InternetService_No    -0.648988 Lower Churn Risk
 cat__StreamingMovies_No internet service    -0.648988 Lower Churn Risk
                     cat__PhoneService_No    -0.551242 Lower Churn Risk
      cat__MultipleLines_No ph

In [29]:
# ------------------------------------------------------------
# 18. SAVE BEST MODEL
# ------------------------------------------------------------

import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(
    best_model,
    "models/churn_tuned_model.pkl"
)

print("\nTuned model saved successfully!")
print("File: models/churn_tuned_model.pkl")


Tuned model saved successfully!
File: models/churn_tuned_model.pkl
